# 🐭 go or nogo? — the real neuron, step by step

**Time:** ~40 minutes &nbsp;•&nbsp; **Goal:** do notebook 1 again — **the same tasks, in the same order** — on the **real** recording, and answer the experiment's question.

## The experiment

This data comes from **Miguel Maravall's lab here at Sussex** ([Bale et al., 2021, *Current Biology*](https://doi.org/10.1016/j.cub.2020.10.059)). A mouse feels one of two whisker "words" — **go** (lick, and get water) or **nogo** (don't lick) — while a microscope records one neuron's activity.

**The question:** does this neuron tell go from nogo?

## How this notebook works

Every task here is a task from notebook 1, with the same variable names. 📎 Keep notebook 1 open next to this one: your answer there is your starting point here. Only these things change:

| | notebook 1 | here |
|---|---|---|
| frames | 50 | 21,238 |
| each word lasts | 3 frames | **9** frames → `[::9]` |
| window | 2 before, 5 after | **10** before, **30** after → `dff[start - 10 : start + 30]` |
| skip trials too near the start | `if start >= 2` | `if start >= 10` |
| peak after the start | `go_response[2:]` | `go_response[10:]` |
| rounding | 1–2 decimals | **3** decimals — the real numbers are small |

Same rhythm as notebook 1: 📘 **Demo** — 🔬 **Write it** with a **self-check** — 🐞 **Broken**.

## Our data — the real neuron

Run this first. It loads the same three arrays as notebook 1, one number per frame — `dff`, `go` and `nogo` — but now from the lecture's recording.

The cell prints frames 40 to 55. Look at frame 43: the first go word starts there and plays for **9** frames.

In [ ]:
# Run this first. It loads the lecture neuron's recording —
# from data/ if you downloaded it, otherwise straight from GitHub.
import urllib.request
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

DATA_URL = "https://raw.githubusercontent.com/HocheggerLab/y3-bio-python/main/lab05/data/"
DATA = Path("data")            # same layout as a uv-start project
DATA.mkdir(exist_ok=True)
NEURON_1 = DATA / "neuron_0588_20180828_5_lecture.csv"
NEURON_2 = DATA / "neuron_0406_20180620_8_lecture.csv"
for path in [NEURON_1, NEURON_2]:
    if not path.exists():
        urllib.request.urlretrieve(DATA_URL + path.name, path)

# Columns: 0 frame, 1 dff, 2 go, 3 nogo, 4 reward, 5 lick
dff  = np.loadtxt(NEURON_1, delimiter=",", skiprows=1, usecols=1)
go   = np.loadtxt(NEURON_1, delimiter=",", skiprows=1, usecols=2, dtype=int)
nogo = np.loadtxt(NEURON_1, delimiter=",", skiprows=1, usecols=3, dtype=int)

FRAME_RATE = 10.8   # frames per second

print("frame:", np.arange(40, 56))
print("dff:  ", dff[40:56].round(1))
print("go:   ", go[40:56])
print("nogo: ", nogo[40:56])

## Step 1 — Meet the arrays &nbsp; 📎 *notebook 1, step 1*

The same attributes and methods — the array is just bigger:

```python
dff.shape      # attribute: (21238,)
dff.mean()     # method: the average
dff.max()      # the largest value
dff.argmax()   # WHERE the largest value is
```

In [ ]:
# DEMO - ask the array about itself
print("shape:", dff.shape)
print("mean: ", round(dff.mean(), 3))
print("max:  ", dff.max(), "at frame", dff.argmax())
print("x 100:", (dff * 100)[:6])        # every value, at once

### 🔬 Write it

**Your task:**

1. Store how many frames there are in `n_frames` — use an attribute.
2. Store the **lowest** value of `dff` in `lowest`, and **where** it is in `lowest_frame`.

> 📎 Exactly your notebook 1 answer.

In [ ]:
# TODO — n_frames, lowest and lowest_frame


# --- self-check (do not edit) ---
assert "n_frames" in globals(), "❌ No variable called n_frames yet"
assert "lowest" in globals() and "lowest_frame" in globals(), "❌ Make lowest and lowest_frame"
assert n_frames == 21238, f"❌ n_frames should be 21238, got {n_frames} — dff.size"
assert lowest == dff.min(), f"❌ lowest should be {dff.min()}, got {lowest} — dff.min()"
assert lowest_frame != lowest, "❌ lowest_frame is the lowest VALUE — you want its POSITION: dff.argmin()"
assert lowest_frame == 13647, f"❌ lowest_frame should be 13647, got {lowest_frame} — dff.argmin()"
print("✅ Correct — the lowest point is", lowest, "at frame", lowest_frame)

### 🐞 Broken — the brackets matter

The same bug as notebook 1. Run it, read the **last line** of the error, then fix it.

In [ ]:
# 🐞 BROKEN — the peak, rounded to 1 decimal
peak = dff.max
print("peak:", round(peak, 1))


# --- self-check (do not edit) ---
assert not callable(peak), "❌ peak is still the method itself — add brackets to run it: dff.max()"
assert peak == dff.max(), f"❌ peak should be {dff.max()}, got {peak}"
print("✅ Fixed — dff.max is the method; dff.max() is its answer")

## Step 2 — A first try: a mask &nbsp; 📎 *notebook 1, step 2*

What does the neuron do **while** the go word plays? `go == 1` is the mask; `dff[go == 1]` keeps only those frames.

In [ ]:
# DEMO - activity while go is on
mask = go == 1
print("dff while go is on:", dff[mask])
print("their mean:", round(dff[mask].mean(), 3))
print("mean overall:", round(dff.mean(), 3))

### 🔬 Write it

**Your task:** store the mean of `dff` **while nogo is on**, rounded to **3** decimal places, in `nogo_on_mean`.

> 📎 Your notebook 1 answer — round to 3 instead of 2.

In [ ]:
# TODO — the mean of dff while nogo is on, rounded to 3 decimals


# --- self-check (do not edit) ---
assert "nogo_on_mean" in globals(), "❌ No variable called nogo_on_mean yet"
assert nogo_on_mean != round(dff.mean(), 3), "❌ That's the mean of ALL frames — filter with the mask first: dff[nogo == 1]"
assert nogo_on_mean == 0.007, f"❌ nogo_on_mean should be 0.007, got {nogo_on_mean} — dff[nogo == 1].mean(), rounded to 3"
print("✅ Correct — about 0 for both words, below the overall mean of 0.058. The mask finds nothing!")

**Both words give about 0** — the same surprise as notebook 1. The real neuron also answers **after** the word, not while it plays. So, as before: find where each word starts, and look at a window round it.

## Step 3 — Where is the word on? `np.where` &nbsp; 📎 *notebook 1, step 3*

`np.where(go == 1)[0]` turns the mask into frame numbers. `[0]` takes the array out of the tuple.

In [ ]:
# DEMO - the frame numbers where go is on
go_on = np.where(go == 1)[0]
print(go_on[:18])
print(len(go_on), "frames with go on")

Blocks of **nine** now: `43 … 51`, `125 … 133` — one block per trial.

### 🐞 Broken — the tuple trap

This should count the frames where **nogo** is on — that's 2376. No error, just a wrong answer.

> 📎 The same fix as notebook 1.

In [ ]:
# 🐞 BROKEN — every frame where nogo is on (there are 2376)
nogo_on = np.where(nogo == 1)
print(len(nogo_on), "frames")


# --- self-check (do not edit) ---
assert len(nogo_on) == 2376, f"❌ len(nogo_on) is {len(nogo_on)} — np.where hands back a TUPLE with one array inside. Take it out with [0]"
assert nogo_on[0] == 2, f"❌ The first nogo frame should be 2, got {nogo_on[0]}"
print("✅ Fixed —", len(nogo_on), "frames: 264 nogo words of 9 frames each")

## Step 4 — Where does each trial start? &nbsp; 📎 *notebook 1, step 4*

Each word lasts **9** frames, so the starts are every **9th** position: `go_on[::9]`.

In [ ]:
# DEMO - the first frame of each go trial
go_starts = go_on[::9]
print(go_starts[:5])
print(len(go_starts), "go trials")

### 🔬 Write it

**Your task:** make `nogo_starts` — the first frame of each **nogo** trial — from `nogo_on`.

Then look at the first one. Remember it: it will matter in Step 6.

> 📎 Your notebook 1 answer, with `9` instead of `3`.

In [ ]:
# TODO — nogo_starts, from nogo_on


# --- self-check (do not edit) ---
assert "nogo_starts" in globals(), "❌ No variable called nogo_starts yet"
assert len(nogo_starts) != 2376, "❌ That's every nogo frame — keep every 9th with [::9]"
assert len(nogo_starts) != 792, "❌ Every 3rd? In the real data each word lasts 9 frames: [::9]"
assert len(nogo_starts) == 264, f"❌ There should be 264 nogo trials, got {len(nogo_starts)}"
assert nogo_starts[0] == 2, f"❌ The first nogo trial should start at frame 2, got {nogo_starts[0]}"
print("✅ Correct — 264 nogo trials, and the first starts at frame 2, right at the beginning of the recording")

## Step 5 — Cut one window round a start &nbsp; 📎 *notebook 1, step 5*

**10** frames before the start, **30** from it on — 40 frames, about 3.7 seconds:

```python
dff[start - 10 : start + 30]
```

In the window, the word starts at **position 10**.

In [ ]:
# DEMO - the window round the first go trial
start = go_starts[0]
window = dff[start - 10 : start + 30]
print("frames", start - 10, "to", start + 29, "-", window.size, "frames")
print(window.round(2))

### 🔬 Write it

**Your task:** cut the window round the **second** go trial and store it in `second_window`.

> 📎 Your notebook 1 answer, with `10` and `30` instead of `2` and `5`.

In [ ]:
# TODO — the window round the second go trial


# --- self-check (do not edit) ---
assert "second_window" in globals(), "❌ No variable called second_window yet"
assert len(second_window) == 40, f"❌ A window should be 40 frames, got {len(second_window)} — dff[start - 10 : start + 30]"
assert np.allclose(second_window, dff[115:155]), "❌ second_window should be frames 115 to 154 — use go_starts[1]"
print("✅ Correct — frames 115 to 154")

## Step 6 — Cut them all, and stack them into a grid &nbsp; 📎 *notebook 1, step 6*

The same loop with the real sizes. The grid now has **254 rows** (trials) and **40 columns** (moments).

In [ ]:
# DEMO - every go window, stacked
go_trials = []
for start in go_starts:
    go_trials.append(dff[start - 10 : start + 30])
go_trials = np.array(go_trials)          # a list of 254 windows -> a (254, 40) grid

print("shape:               ", go_trials.shape)
print("trial 1:             ", go_trials[0].shape)
print("column 10, all trials:", go_trials[:, 10].shape)

### 🐞 Broken — the same loop, for nogo

It crashes with an **inhomogeneous shape**, exactly like notebook 1.

> 💡 The first nogo trial starts at frame **2**, so its window is `dff[2 - 10 : 2 + 30]` = `dff[-8:32]` — **empty**.
>
> 📎 The notebook 1 fix — but here a trial needs **10** frames before it: `if start >= 10:`.

In [ ]:
# 🐞 BROKEN — every nogo window, stacked
nogo_trials = []
for start in nogo_starts:
    nogo_trials.append(dff[start - 10 : start + 30])
nogo_trials = np.array(nogo_trials)
print(nogo_trials.shape)


# --- self-check (do not edit) ---
assert nogo_trials.shape == (263, 40), f"❌ nogo_trials should be (263, 40) — 263 trials once the one at frame 2 is skipped — got {nogo_trials.shape}"
print("✅ Fixed — the trial at frame 2 had no 10 frames before it, so it is left out")

## Step 7 — Average down the columns: `axis=0` &nbsp; 📎 *notebook 1, step 7*

| | collapses | answer | here |
|---|---|---|---|
| `.mean(axis=0)` | down the **columns** | one number per **moment** — the average trial | 40 numbers |
| `.mean(axis=1)` | along the **rows** | one number per **trial** | 254 numbers |
| `.mean()` | everything | one number | 1 number |

In [ ]:
# DEMO - the three ways to average the go grid
print("axis=0:", go_trials.mean(axis=0).shape)    # the average trial
print("axis=1:", go_trials.mean(axis=1).shape)    # one per trial
print("all:   ", round(go_trials.mean(), 3))

### 🔬 Write it

**Your task:**

1. Make `go_response` — the **average go trial**: one number per moment.
2. Store its highest value **from the start onwards** (position **10** on — skip the baseline), rounded to **3** decimal places, in `go_peak`.

> 📎 Your notebook 1 answer, with `[10:]` and 3 decimals.

In [ ]:
# TODO — go_response (the average trial) and go_peak


# --- self-check (do not edit) ---
assert "go_response" in globals() and "go_peak" in globals(), "❌ Make go_response and go_peak"
assert np.shape(go_response) != (254,), "❌ go_response has one value per TRIAL — that's axis=1. You want one per moment: axis=0"
assert np.shape(go_response) == (40,), f"❌ go_response should have 40 values, one per moment — go_trials.mean(axis=0). Got shape {np.shape(go_response)}"
assert go_peak == 0.29, f"❌ go_peak should be 0.29, got {go_peak} — go_response[10:].max(), rounded to 3"
print("✅ Correct — the average go trial peaks at", go_peak)

## Step 8 — go against nogo: the answer &nbsp; 📎 *notebook 1, step 8*

**Your task:**

1. Make `nogo_response` — the average **nogo** trial.
2. Store its peak from the start onwards, rounded to **3** decimal places, in `nogo_peak`.
3. Store how many times bigger the go peak is in `ratio`, rounded to 1 decimal place.

In [ ]:
# TODO — nogo_response, nogo_peak and ratio


# --- self-check (do not edit) ---
assert "nogo_response" in globals() and "nogo_peak" in globals() and "ratio" in globals(), "❌ Make nogo_response, nogo_peak and ratio"
assert np.shape(nogo_response) == (40,), f"❌ nogo_response should have 40 values — nogo_trials.mean(axis=0). Got shape {np.shape(nogo_response)}"
assert nogo_peak == 0.057, f"❌ nogo_peak should be 0.057, got {nogo_peak} — nogo_response[10:].max(), rounded to 3"
assert ratio == 5.1, f"❌ ratio should be 5.1, got {ratio} — go_peak / nogo_peak"
print("✅ Puzzle solved — the real neuron responds five times more after go than after nogo. It tells them apart.")

Here are the two answers as curves — plus one single trial, in grey. (Plotting comes in Lab 7 — just run it.)

In [ ]:
# Run me - one trial, and the two averages
seconds = np.arange(-10, 30) / FRAME_RATE    # frames from the start, as seconds

plt.figure(figsize=(8, 3.5))
plt.plot(seconds, go_trials[0], color="lightgrey", label="one go trial")
plt.plot(seconds, go_response, lw=2.5, label=f"go — {len(go_trials)} trials averaged")
plt.plot(seconds, nogo_response, lw=2.5, label=f"nogo — {len(nogo_trials)} trials averaged")
plt.axvline(0, color="black", lw=1)                 # the word starts
plt.axvspan(0, 9 / FRAME_RATE, alpha=0.1)           # ...and plays for 9 frames
plt.xlabel("seconds from the start of the word")
plt.ylabel("activity (dF/F)")
plt.legend()
plt.show()

**Read the picture.** The grey single trial is mostly noise. The averages are not: after **go**, activity climbs as the word ends and stays up; after **nogo** it stays flat. This neuron tells the two words apart — and the mask in Step 2 missed it because it only looked while the word was playing.

> 🧠 **A scientist's doubt.** On go trials the mouse also **licks** and gets **water**. Is the neuron answering the *word*, or the *licking*? This analysis can't tell yet.

## 🔭 Explore further (optional)

- **One number per trial.** `go_trials[:, 10:].mean(axis=1)` gives each trial's average response. Does the neuron respond less towards the end of the session?
- **Words or licks?** Load the `lick` column (`usecols=5`, `dtype=int`) and compare `dff[lick == 1].mean()` with `dff.mean()`.
- **A second neuron.** The **advanced** notebook wraps these steps in a function and runs them on neuron 2 — which gives a very different answer.

## ✅ You've got it if you can…

- [ ] Say the difference between `dff.max()` and `dff.argmax()`, and between `dff.max` and `dff.max()`
- [ ] Filter an array with a mask — and explain why the mask missed the response
- [ ] Turn a mask into positions with `np.where(...)[0]`
- [ ] Explain why `[::9]` gives the start of each trial here, and `[::3]` in notebook 1
- [ ] Cut a window round a start, and explain why the trial at frame 2 broke the stacking
- [ ] Stack windows into a grid, and say what `.mean(axis=0)` and `.mean(axis=1)` give